# Batch Feature Extractor (Voronoi Segmentation)
This notebook isolates specific feature point clouds using Nearest-Neighbor (Voronoi) segmentation, calculates their exact Chamfer Distance noise without distance truncation, and saves them to disk to serve as the training dataset for PointNet.

In [33]:
# ==========================================
# 1. INTERACTIVE VISUALIZATION & CHAMFER ANALYSIS
# ==========================================
# Run this cell on a single viewpoint to visually verify:
#   Window 1: All surfaces Voronoi segmentation (multi-color palette + CAD references).
#   Window 2: Specific feature inspection (Raycasted Ideal in Green vs Actual Noisy in Blue).
import math
import os
import glob
import copy
import numpy as np
import open3d as o3d

WORKPIECE = "workpiece31"
EXPERIMENT = "test_10_realgrasp" 

# WORKPIECE = "TH0011AV"
# EXPERIMENT = "test_8_simulation2"  


VIEWPOINT_IDX = 202
NUMBER_OF_POINTS = 5000
SPECIFIC_FEATURE = "feature2"     # e.g., 'feature0', 'feature1', 'feature2', 'surface0'

print(f"Loading {WORKPIECE} - Viewpoint {VIEWPOINT_IDX} for interactive validation...")

PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{WORKPIECE}"
SIMULATION_DIR = f"simulation/{EXPERIMENT}/{WORKPIECE}"
SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE}"
if not os.path.exists(SIM_DIR):
    SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}"
WORKPIECE_DIR = f"workpiece/{WORKPIECE}"

noisy_pcd_path = os.path.join(SIMULATION_DIR, f"viewpoint_simulated_noise_{VIEWPOINT_IDX}.pcd")
if not os.path.exists(noisy_pcd_path):
    noisy_pcd_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_noise_{VIEWPOINT_IDX}.pcd")
perfect_pcd_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{VIEWPOINT_IDX}.pcd")

def default_visualization(geometries, window_name="Default Visualization", zoom=1.0):
    azimuth_deg = -45
    elevation_deg = -135
    az = math.radians(azimuth_deg)
    el = math.radians(elevation_deg)
    front = np.array([math.cos(el) * math.cos(az), math.cos(el) * math.sin(az), math.sin(el)])
    front = -front
    if isinstance(geometries, list) and len(geometries) > 0:
        lookat = geometries[0].get_center()
    else:
        lookat = [0, 0, 0]
    up = [0, 0, 1]
    o3d.visualization.draw_geometries(geometries, window_name=window_name, width=1024, height=768, lookat=lookat, up=up, front=front, zoom=zoom)

try:
    # 1. Load full workpiece CAD model
    workpiece_mesh = o3d.io.read_triangle_mesh(os.path.join(WORKPIECE_DIR, "workpiece.stl"))
    workpiece_mesh.compute_vertex_normals()
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS * 4)

    # 2. Load all target features
    feature_files = sorted(glob.glob(os.path.join(WORKPIECE_DIR, "feature*.stl")))
    if not feature_files:
        feature_files = sorted(glob.glob(os.path.join(WORKPIECE_DIR, "surface*.stl")))
        
    feature_pcds = []
    feature_names = []
    for f in feature_files:
        mesh = o3d.io.read_triangle_mesh(f)
        mesh.compute_vertex_normals()
        pcd = mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
        feature_pcds.append(pcd)
        feature_names.append(os.path.basename(f))

    # 3. Background extraction
    dists_to_features = np.ones(len(workpiece_pcd.points)) * 999.0
    for fpcd in feature_pcds:
        dists = np.asarray(workpiece_pcd.compute_point_cloud_distance(fpcd))
        dists_to_features = np.minimum(dists_to_features, dists)
    
    background_indices = np.where(dists_to_features > 2.0)[0]
    background_pcd = workpiece_pcd.select_by_index(background_indices)
    
    classes_pcd = feature_pcds.copy()
    class_names = feature_names.copy()
    if len(background_pcd.points) > 0:
        classes_pcd.append(background_pcd)
        class_names.append("Background (Outer Wall)")

    # 4. Load Scanned & Raycasted PCDs
    noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)
    if len(noisy_pcd.points) == 0:
        raise ValueError(f"Could not load PCD from {noisy_pcd_path}")

    has_perfect_raycast = os.path.exists(perfect_pcd_path)
    if has_perfect_raycast:
        perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)

    # 5. Voronoi Segmentation for Noisy PCD
    print("\nRunning Voronoi Multi-Surface Segmentation...")
    dists_noisy = np.zeros((len(noisy_pcd.points), len(classes_pcd)))
    for i, cls_pcd in enumerate(classes_pcd):
        dists_noisy[:, i] = np.asarray(noisy_pcd.compute_point_cloud_distance(cls_pcd))
    assignments_noisy = np.argmin(dists_noisy, axis=1)

    # Voronoi Segmentation for Perfect Raycasted PCD
    if has_perfect_raycast:
        dists_perf = np.zeros((len(perfect_pcd.points), len(classes_pcd)))
        for i, cls_pcd in enumerate(classes_pcd):
            dists_perf[:, i] = np.asarray(perfect_pcd.compute_point_cloud_distance(cls_pcd))
        assignments_perf = np.argmin(dists_perf, axis=1)

    # 6. Print Multi-Surface Summary
    print("=" * 75)
    print(f"1. VORONOI SEGMENTATION SUMMARY (ALL SURFACES)")
    print(f"Workpiece: {WORKPIECE} | Viewpoint: {VIEWPOINT_IDX} | Experiment: {EXPERIMENT}")
    print("=" * 75)

    for i, name in enumerate(feature_names):
        idx = np.where(assignments_noisy == i)[0]
        is_target = (SPECIFIC_FEATURE and (SPECIFIC_FEATURE in name or SPECIFIC_FEATURE == name))
        marker = "-> [INSPECTED]" if is_target else "   [Feature]  "
        print(f"{marker} {name:<14} | Captured: {len(idx):>5} points")

    bg_idx = np.where(assignments_noisy == len(feature_names))[0]
    print(f"   [Ignored]  Background     | Captured: {len(bg_idx):>5} points")
    print("=" * 75)

    # 7. Identify target feature for Window 2
    target_feature_idx = 0
    if SPECIFIC_FEATURE:
        for idx, name in enumerate(feature_names):
            if SPECIFIC_FEATURE in name or SPECIFIC_FEATURE == name:
                target_feature_idx = idx
                break
    
    target_name = feature_names[target_feature_idx]
    target_noisy_indices = np.where(assignments_noisy == target_feature_idx)[0]
    feature_noisy_pcd = noisy_pcd.select_by_index(target_noisy_indices)
    feature_noisy_pcd.paint_uniform_color([0.0, 0.4, 1.0]) # Blue for Actual Scan

    if has_perfect_raycast:
        target_perf_indices = np.where(assignments_perf == target_feature_idx)[0]
        feature_perf_pcd = perfect_pcd.select_by_index(target_perf_indices)
        ref_label = "Simulated Raycast"
    else:
        feature_perf_pcd = feature_pcds[target_feature_idx]
        ref_label = "CAD Model (Fallback)"

    feature_perf_pcd.paint_uniform_color([0.0, 0.8, 0.2]) # Green for Ideal Raycast

    # Chamfer Distance calculation
    if len(feature_noisy_pcd.points) > 0 and len(feature_perf_pcd.points) > 0:
        d_s2c = np.mean(feature_noisy_pcd.compute_point_cloud_distance(feature_perf_pcd))
        d_c2s = np.mean(feature_perf_pcd.compute_point_cloud_distance(feature_noisy_pcd))
        cd_val = d_s2c + d_c2s
        print(f"\n2. FEATURE CHAMFER NOISE ANALYSIS ({target_name}):")
        print(f"   Actual Scan: {len(feature_noisy_pcd.points)} pts | Ideal ({ref_label}): {len(feature_perf_pcd.points)} pts")
        print(f"   Chamfer Distance: {cd_val:.4f} mm (Scan->Ideal: {d_s2c:.4f}, Ideal->Scan: {d_c2s:.4f})")
    else:
        cd_val = 0.0
        print(f"\n2. FEATURE CHAMFER NOISE ANALYSIS ({target_name}): No points captured from this viewpoint.")

    # ---------------------------------------------------------
    # WINDOW 1: All Surfaces' Voronoi Segmentation
    # ---------------------------------------------------------
    print("\n>>> Launching Window 1: All Surfaces Voronoi Segmentation...")
    print(">>> [NOTE] Close Window 1 to automatically open Window 2!")

    geometries_1 = []
    colors = [[1.0, 0.2, 0.2], [0.0, 0.8, 0.2], [0.0, 0.4, 1.0], [1.0, 0.8, 0.0], [0.8, 0.2, 0.8], [0.2, 0.8, 0.8]]
    
    for i, name in enumerate(class_names):
        idx = np.where(assignments_noisy == i)[0]
        assigned_pcd = noisy_pcd.select_by_index(idx)
        
        if "Background" in name:
            color = [0.5, 0.5, 0.5] # Gray
        else:
            color = colors[i % len(colors)]
            
        assigned_pcd.paint_uniform_color(color)
        geometries_1.append(assigned_pcd)
        
        # Reference feature shifted to left
        if "Background" not in name:
            ref_pcd = copy.deepcopy(classes_pcd[i])
            ref_pcd.paint_uniform_color(color)
            ref_pcd.translate([-125, 0, 0])
            geometries_1.append(ref_pcd)

    default_visualization(geometries_1, window_name=f"1. All Surfaces Voronoi Segmentation - {WORKPIECE} View {VIEWPOINT_IDX}")

    # ---------------------------------------------------------
    # WINDOW 2: Raycasted Ideal (Green) vs Actual Noisy Scan (Blue)
    # ---------------------------------------------------------
    print(f"\n>>> Launching Window 2: {target_name} (Raycasted vs Actual)...")
    geometries_2 = [feature_perf_pcd, feature_noisy_pcd]
    win2_title = f"2. {target_name} Inspection - Green: Ideal ({ref_label}) | Blue: Actual Scan (CD: {cd_val:.2f}mm)"
    default_visualization(geometries_2, window_name=win2_title)

except Exception as e:
    print(f"Error loading or visualizing point clouds: {e}")


Loading workpiece31 - Viewpoint 202 for interactive validation...

Running Voronoi Multi-Surface Segmentation...
1. VORONOI SEGMENTATION SUMMARY (ALL SURFACES)
Workpiece: workpiece31 | Viewpoint: 202 | Experiment: test_10_realgrasp
   [Feature]   feature0.stl   | Captured: 21000 points
   [Feature]   feature1.stl   | Captured:  2891 points
-> [INSPECTED] feature2.stl   | Captured: 22073 points
   [Feature]   feature3.stl   | Captured:  4311 points
   [Ignored]  Background     | Captured:   105 points

2. FEATURE CHAMFER NOISE ANALYSIS (feature2.stl):
   Actual Scan: 22073 pts | Ideal (Simulated Raycast): 1568 pts
   Chamfer Distance: 14.0047 mm (Scan->Ideal: 10.4206, Ideal->Scan: 3.5841)

>>> Launching Window 1: All Surfaces Voronoi Segmentation...
>>> [NOTE] Close Window 1 to automatically open Window 2!

>>> Launching Window 2: feature2.stl (Raycasted vs Actual)...


In [3]:
# ==========================================
# 2. BATCH PROCESSING LOOP
# ==========================================
import os
import glob
import pandas as pd
import numpy as np
import open3d as o3d

WORKPIECES = ["workpiece31"]
# WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV", "TH0041AV", "TH0042AV", "TH0051AV", "TH0052AV", "TH0061AV", "TH0062AV", "TH0071AV", "TH0072AV"]
EXPERIMENT = "test_10_realgrasp"
DATASET_NAME = EXPERIMENT
NUMBER_OF_POINTS = 5000

results = []

for workpiece in WORKPIECES:
    print(f"\n==========================================")
    print(f"ANALYZING & EXTRACTING FEATURES: {workpiece}")
    print(f"==========================================")
    
    PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{workpiece}"
    SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{workpiece}"
    WORKPIECE_DIR = f"workpiece/{workpiece}"
    
    EXPORT_DIR = PROCESSED_DIR
    os.makedirs(EXPORT_DIR, exist_ok=True)
    
    if not os.path.exists(PROCESSED_DIR):
        print(f"Skipping {workpiece} - No processed data found.")
        continue
        
    # 1. Load full workpiece CAD model
    workpiece_mesh = o3d.io.read_triangle_mesh(os.path.join(WORKPIECE_DIR, "workpiece.stl"))
    workpiece_mesh.compute_vertex_normals()
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS * 4)
        
    # 2. Load all target features (supports feature*.stl and fallback to surface*.stl)
    feature_files = sorted(glob.glob(os.path.join(WORKPIECE_DIR, "feature*.stl")))
    if not feature_files:
        feature_files = sorted(glob.glob(os.path.join(WORKPIECE_DIR, "surface*.stl")))
    if not feature_files:
        print(f"No feature files found for {workpiece}.")
        continue
        
    feature_pcds = []
    feature_names = []
    for f in feature_files:
        mesh = o3d.io.read_triangle_mesh(f)
        mesh.compute_vertex_normals()
        pcd = mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
        feature_pcds.append(pcd)
        feature_names.append(os.path.basename(f))
        
    # 3. Generate Background PCD
    dists_to_features = np.ones(len(workpiece_pcd.points)) * 999.0
    for fpcd in feature_pcds:
        dists = np.asarray(workpiece_pcd.compute_point_cloud_distance(fpcd))
        dists_to_features = np.minimum(dists_to_features, dists)
    
    background_indices = np.where(dists_to_features > 2.0)[0]
    background_pcd = workpiece_pcd.select_by_index(background_indices)
    
    classes_pcd = feature_pcds.copy()
    class_names = feature_names.copy()
    if len(background_pcd.points) > 0:
        classes_pcd.append(background_pcd)
        class_names.append("Background (Outer Wall)")

    # Find all scanned/simulated point clouds
    SIMULATION_OUTPUT_DIR = f"simulation/{EXPERIMENT}/{workpiece}"
    pcd_files = glob.glob(os.path.join(SIMULATION_OUTPUT_DIR, "viewpoint_simulated_noise_*.pcd"))
    if not pcd_files:
        pcd_files = glob.glob(os.path.join(PROCESSED_DIR, "viewpoint_simulated_noise_*.pcd"))
    pcd_files = [f for f in pcd_files if "_feature" not in f and "_surface" not in f]
    
    # Sort files numerically instead of lexicographically
    import re
    def extract_number(filename):
        match = re.search(r'viewpoint_simulated_noise_(\d+)\.pcd', filename)
        return int(match.group(1)) if match else -1
    pcd_files = sorted(pcd_files, key=extract_number)
    
    # Track how many we saved for print logs
    count_hits = {name: 0 for name in feature_names}
        
    for noisy_pcd_path in pcd_files:
        viewpoint_name_noisy = os.path.basename(noisy_pcd_path).replace('.pcd', '')
        view_idx = viewpoint_name_noisy.replace('viewpoint_simulated_noise_', '')
        
        perfect_pcd_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{view_idx}.pcd")
        # If perfect raycast does not exist (e.g. real data), use CAD feature as reference geometry
        has_perfect_raycast = os.path.exists(perfect_pcd_path)
        
        noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)
        if has_perfect_raycast:
            perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)
            # Voronoi Segmentation for Perfect PCD
            dists_perf = np.zeros((len(perfect_pcd.points), len(classes_pcd)))
            for i, cls_pcd in enumerate(classes_pcd):
                dists_perf[:, i] = np.asarray(perfect_pcd.compute_point_cloud_distance(cls_pcd))
            assignments_perf = np.argmin(dists_perf, axis=1)
        
        # Voronoi Segmentation for Noisy PCD
        dists_noisy = np.zeros((len(noisy_pcd.points), len(classes_pcd)))
        for i, cls_pcd in enumerate(classes_pcd):
            dists_noisy[:, i] = np.asarray(noisy_pcd.compute_point_cloud_distance(cls_pcd))
        assignments_noisy = np.argmin(dists_noisy, axis=1)
        
        # Process each target feature (skip background)
        for feature_idx in range(len(feature_names)):
            feature_name = feature_names[feature_idx]
            feature_cad = feature_pcds[feature_idx]
            
            # Extract Noisy Feature
            noisy_indices = np.where(assignments_noisy == feature_idx)[0]
            if len(noisy_indices) < 10:
                continue
            feature_noisy_pcd = noisy_pcd.select_by_index(noisy_indices)
            
            # Calculate Chamfer Distance (UNTRUNCATED!)
            dists_s2c = np.asarray(feature_noisy_pcd.compute_point_cloud_distance(feature_cad))
            dists_c2s = np.asarray(feature_cad.compute_point_cloud_distance(feature_noisy_pcd))
            if len(dists_c2s) == 0:
                continue
            chamfer_dist = np.mean(dists_s2c) + np.mean(dists_c2s)
            
            # Extract or Generate Perfect Feature for PointNet
            if has_perfect_raycast:
                perf_indices = np.where(assignments_perf == feature_idx)[0]
                if len(perf_indices) < 10:
                    continue
                feature_perf_pcd = perfect_pcd.select_by_index(perf_indices)
            else:
                feature_perf_pcd = feature_noisy_pcd
            
            count_hits[feature_name] += 1
            
            # Export Feature PCD for PointNet
            feature_clean_name = feature_name.replace('.stl', '')
            export_filename = f"viewpoint_simulated_{view_idx}_{feature_clean_name}.pcd"
            export_path = os.path.join(EXPORT_DIR, export_filename)
            o3d.io.write_point_cloud(export_path, feature_perf_pcd)
            
            # Save to CSV
            relative_path = f"{workpiece}/{export_filename}"
            results.append({
                "filename": relative_path,
                "dist_s2r": np.mean(dists_s2c),
                "dist_r2s": np.mean(dists_c2s),
                "chamfer_value": chamfer_dist,
                "asymmetry_value": 0.0
            })
            
    for feature_name, hits in count_hits.items():
        print(f"--> {feature_name}: Exported {hits} cropped feature point clouds")

print("\n==========================================")
print("DONE EXTRACTING FEATURES!")
print("==========================================")

df_results = pd.DataFrame(results)
CSV_EXPORT_DIR = f"processed_data/{DATASET_NAME}"
os.makedirs(CSV_EXPORT_DIR, exist_ok=True)
csv_path = os.path.join(CSV_EXPORT_DIR, "metadata.csv")
df_results.to_csv(csv_path, index=False)
print(f"Saved {len(df_results)} point cloud labels to {csv_path}")



ANALYZING & EXTRACTING FEATURES: workpiece31
--> surface0.stl: Exported 287 cropped feature point clouds
--> surface1.stl: Exported 287 cropped feature point clouds
--> surface2.stl: Exported 287 cropped feature point clouds
--> surface3.stl: Exported 287 cropped feature point clouds

DONE EXTRACTING FEATURES!
Saved 1148 point cloud labels to processed_data/test_10_realgrasp\metadata.csv


In [18]:
# ==========================================
# INTERACTIVE DEBUGGING VISUALIZATION (TWO WINDOWS)
# ==========================================
import math
import os
import glob
import numpy as np
import open3d as o3d

# --- CONFIGURATION ---



WORKPIECE = "TH0011AV"
EXPERIMENT = "test_8_simulation2"

# WORKPIECE = "workpiece31"
# EXPERIMENT = "test_10_realgrasp"

VIEWPOINT_IDX = 202
SURFACE_USED = "feature2"  # e.g., 'surface0', 'surface1', 'surface2', 'surface3' or 'feature*'
NUMBER_OF_POINTS = 5000
RAYCAST_ON_THE_FLY = False   # Set to True to always generate single-perspective visible raycast via HPR

print(f"\nLoading {WORKPIECE} - Viewpoint {VIEWPOINT_IDX} ({SURFACE_USED}) for interactive validation...")

PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{WORKPIECE}"
SIMULATION_DIR = f"simulation/{EXPERIMENT}/{WORKPIECE}"
SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{WORKPIECE}"
if not os.path.exists(SIM_DIR):
    SIM_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}"
WORKPIECE_DIR = f"workpiece/{WORKPIECE}"

noisy_pcd_path = os.path.join(SIMULATION_DIR, f"viewpoint_simulated_noise_{VIEWPOINT_IDX}.pcd")
if not os.path.exists(noisy_pcd_path):
    noisy_pcd_path = os.path.join(PROCESSED_DIR, f"viewpoint_simulated_noise_{VIEWPOINT_IDX}.pcd")
perfect_pcd_path = os.path.join(SIM_DIR, f"viewpoint_simulated_{VIEWPOINT_IDX}.pcd")

# Resolve surface CAD path
target_surface_path = os.path.join(WORKPIECE_DIR, f"{SURFACE_USED}.stl")
if not os.path.exists(target_surface_path):
    alt_name = SURFACE_USED.replace('feature', 'surface') if 'feature' in SURFACE_USED else SURFACE_USED.replace('surface', 'feature')
    target_surface_path = os.path.join(WORKPIECE_DIR, f"{alt_name}.stl")

try:
    # ---------------------------------------------------------
    # TOP VISUALIZATION: Full CAD Feature (Red) & Full Scanned PCD (Gray)
    # ---------------------------------------------------------
    noisy_pcd = o3d.io.read_point_cloud(noisy_pcd_path)
    if len(noisy_pcd.points) == 0:
        raise ValueError(f"Could not load PCD from {noisy_pcd_path}")
    noisy_pcd.paint_uniform_color([0.5, 0.5, 0.5]) # Gray
    
    mesh = o3d.io.read_triangle_mesh(target_surface_path)
    mesh.compute_vertex_normals()
    feature_cad = mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
    feature_cad.paint_uniform_color([1.0, 0.0, 0.0]) # Red

    # ---------------------------------------------------------
    # BOTTOM VISUALIZATION: Raycasted Ideal (Green) & Raycasted Noisy (Blue)
    # ---------------------------------------------------------
    print("Running Voronoi Segmentation...")
    workpiece_mesh = o3d.io.read_triangle_mesh(os.path.join(WORKPIECE_DIR, "workpiece.stl"))
    workpiece_mesh.compute_vertex_normals()
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS * 4)

    surface_files = sorted(glob.glob(os.path.join(WORKPIECE_DIR, "surface*.stl")))
    if not surface_files:
        surface_files = sorted(glob.glob(os.path.join(WORKPIECE_DIR, "feature*.stl")))
    feature_pcds = []
    target_feature_idx = -1

    for idx, f in enumerate(surface_files):
        m = o3d.io.read_triangle_mesh(f)
        m.compute_vertex_normals()
        p = m.sample_points_poisson_disk(number_of_points=NUMBER_OF_POINTS)
        feature_pcds.append(p)
        if SURFACE_USED in f or (SURFACE_USED.replace('surface', 'feature') in f) or (SURFACE_USED.replace('feature', 'surface') in f):
            target_feature_idx = idx

    if target_feature_idx == -1:
        target_feature_idx = 0

    dists_to_features = np.ones(len(workpiece_pcd.points)) * 999.0
    for fpcd in feature_pcds:
        dists = np.asarray(workpiece_pcd.compute_point_cloud_distance(fpcd))
        dists_to_features = np.minimum(dists_to_features, dists)

    background_indices = np.where(dists_to_features > 2.0)[0]
    background_pcd = workpiece_pcd.select_by_index(background_indices)

    classes_pcd = feature_pcds.copy()
    if len(background_pcd.points) > 0:
        classes_pcd.append(background_pcd)

    # 1. Segment Noisy Scanned PCD
    dists_noisy = np.zeros((len(noisy_pcd.points), len(classes_pcd)))
    for i, cls_pcd in enumerate(classes_pcd):
        dists_noisy[:, i] = np.asarray(noisy_pcd.compute_point_cloud_distance(cls_pcd))
    assignments_noisy = np.argmin(dists_noisy, axis=1)
    noisy_indices = np.where(assignments_noisy == target_feature_idx)[0]
    feature_noisy_pcd = noisy_pcd.select_by_index(noisy_indices)
    feature_noisy_pcd.paint_uniform_color([0.0, 0.0, 1.0]) # Blue

    # 2. Get Visible / Raycasted Ideal Feature Point Cloud
    feature_perf_pcd = None
    pose_path = os.path.join(SIM_DIR, f"viewpoint_pose_{VIEWPOINT_IDX}.npy")

    # Mode 1: Generate Visible Line-of-Sight on the Fly via HPR
    if RAYCAST_ON_THE_FLY and os.path.exists(pose_path):
        camera_pose = np.load(pose_path)
        cam_pos = camera_pose[:3, 3]
        d_cam = np.linalg.norm(cam_pos - feature_cad.get_center())
        _, visible_indices = feature_cad.hidden_point_removal(cam_pos, radius=d_cam * 5.0)
        feature_perf_pcd = feature_cad.select_by_index(visible_indices)
        print(f"👉 [Mode: ON-THE-FLY HPR] Raycasted {len(feature_perf_pcd.points)} visible points from Camera Pose.")

    # Mode 2: Load pre-computed simulated raycast from disk
    elif os.path.exists(perfect_pcd_path):
        perfect_pcd = o3d.io.read_point_cloud(perfect_pcd_path)
        dists_perf = np.zeros((len(perfect_pcd.points), len(classes_pcd)))
        for i, cls_pcd in enumerate(classes_pcd):
            dists_perf[:, i] = np.asarray(perfect_pcd.compute_point_cloud_distance(cls_pcd))
        assignments_perf = np.argmin(dists_perf, axis=1)
        perf_indices = np.where(assignments_perf == target_feature_idx)[0]
        if len(perf_indices) > 0:
            feature_perf_pcd = perfect_pcd.select_by_index(perf_indices)
        print(f"👉 [Mode: DISK FILE] Loaded {len(feature_perf_pcd.points)} points from {perfect_pcd_path}")

    # Mode 3: Fallback to full 3D CAD mesh
    if feature_perf_pcd is None:
        feature_perf_pcd = feature_cad
        print(f"👉 [Mode: FULL CAD] Using entire 3D CAD mesh ({len(feature_perf_pcd.points)} points)")

    feature_perf_pcd.paint_uniform_color([0.0, 1.0, 0.0]) # Green

    # Calculate Chamfer Distance on Raycasted Feature
    if len(feature_noisy_pcd.points) > 0 and len(feature_perf_pcd.points) > 0:
        d_s2c = np.mean(feature_noisy_pcd.compute_point_cloud_distance(feature_perf_pcd))
        d_c2s = np.mean(feature_perf_pcd.compute_point_cloud_distance(feature_noisy_pcd))
        cd_val = d_s2c + d_c2s
        print(f"\nChamfer Distance ({SURFACE_USED} View {VIEWPOINT_IDX}): {cd_val:.4f} mm (s2c: {d_s2c:.4f}, c2s: {d_c2s:.4f})")
    
    # ---------------------------------------------------------
    # VISUALIZATION LAUNCHER
    # ---------------------------------------------------------
    def default_visualization(geoms, window_name):
        lookat = geoms[0].get_center()
        up = [0, 0, 1]
        front = [0, -1, 1]
        o3d.visualization.draw_geometries(geoms, window_name=window_name, width=1024, height=768, lookat=lookat, up=up, front=front, zoom=0.8)

    # 1st Window
    print("\nLaunching Window 1 (Full CAD Feature in Red & Full Scanned Scene in Gray) ...")
    print("👉 CLOSE Window 1 to automatically open Window 2!")
    geometries_1 = [noisy_pcd, feature_cad]
    default_visualization(geometries_1, window_name=f"1. Full Scene - {WORKPIECE} View {VIEWPOINT_IDX}")

    # 2nd Window
    print(f"\nLaunching Window 2 (Raycasted Ideal in Green [{len(feature_perf_pcd.points)} pts] vs Raycasted Scanned in Blue [{len(feature_noisy_pcd.points)} pts]) ...")
    geometries_2 = [feature_perf_pcd, feature_noisy_pcd]
    default_visualization(geometries_2, window_name=f"2. Cropped Features - {WORKPIECE} View {VIEWPOINT_IDX} ({SURFACE_USED})")

except Exception as e:
    print(f"Error loading or visualizing point clouds: {e}")



Loading TH0011AV - Viewpoint 202 (feature2) for interactive validation...
Running Voronoi Segmentation...
👉 [Mode: DISK FILE] Loaded 10214 points from viewpoints_candidate/testing_data/test_8_simulation2/TH0011AV\viewpoint_simulated_202.pcd

Chamfer Distance (feature2 View 202): 3.5407 mm (s2c: 1.1621, c2s: 2.3786)

Launching Window 1 (Full CAD Feature in Red & Full Scanned Scene in Gray) ...
👉 CLOSE Window 1 to automatically open Window 2!

Launching Window 2 (Raycasted Ideal in Green [10214 pts] vs Raycasted Scanned in Blue [673 pts]) ...
